# 🤖 TrendSense — Notebook 3: Model Training

**Objective:** Train ARIMA (baseline), Random Forest, and XGBoost models. Compare performance with and without TVI features. Analyze feature importance.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import config
from src.data_ingestion import download_walmart_data, fetch_google_trends
from src.feature_engineering import merge_all_features, get_feature_columns
from src.models import (
    train_arima_baseline, train_random_forest, train_xgboost,
    temporal_train_test_split, compare_models, time_series_cv,
    compute_metrics, save_model
)
from src.utils import plot_model_comparison, plot_prediction_vs_actual, plot_feature_importance

sns.set_theme(style='darkgrid')
print('✅ Libraries loaded')

## 1. Prepare Feature-Engineered Data

In [ ]:
# Load datasets
walmart_df = download_walmart_data()
trends_df = fetch_google_trends()

# Feature engineering
featured_df = merge_all_features(walmart_df, trends_df, category='General')
print(f'\nFeatured dataset: {featured_df.shape}')
featured_df.head()

In [ ]:
# Get feature columns
feature_cols = get_feature_columns(featured_df, 'Weekly_Sales')
print(f'Total features: {len(feature_cols)}')
print(f'\nFeature list:')
for i, col in enumerate(feature_cols, 1):
    tvi_marker = ' ⭐' if 'tvi' in col.lower() or 'spike' in col.lower() or 'trend' in col.lower() else ''
    print(f'  {i:2d}. {col}{tvi_marker}')

## 2. Train-Test Split (Temporal)

In [ ]:
X_train, X_test, y_train, y_test = temporal_train_test_split(
    featured_df, target_col='Weekly_Sales', feature_cols=feature_cols
)

print(f'Training: {X_train.shape[0]} rows, {X_train.shape[1]} features')
print(f'Testing:  {X_test.shape[0]} rows')
print(f'\nTrain period: rows 0-{len(X_train)-1}')
print(f'Test period:  rows {len(X_train)}-{len(X_train)+len(X_test)-1}')

## 3. Model Training

In [ ]:
results = {}

# 3.1 ARIMA Baseline
print('='*60)
train_series = featured_df.sort_values('Date')['Weekly_Sales'].iloc[:len(X_train)]
test_series = featured_df.sort_values('Date')['Weekly_Sales'].iloc[len(X_train):]

arima_result = train_arima_baseline(train_series, test_series)
results['ARIMA (Baseline)'] = arima_result

In [ ]:
# 3.2 Random Forest
print('='*60)
rf_result = train_random_forest(X_train, y_train, X_test, y_test)
results['Random Forest'] = rf_result

In [ ]:
# 3.3 XGBoost + TVI
print('='*60)
xgb_result = train_xgboost(X_train, y_train, X_test, y_test)
results['XGBoost + TVI'] = xgb_result

## 4. Model Comparison

In [ ]:
comparison = compare_models(results)
comparison

In [ ]:
# Visual comparison
fig = make_subplots(rows=1, cols=3, subplot_titles=['MAPE (%) — Lower is Better',
                                                      'RMSE — Lower is Better',
                                                      'R² — Higher is Better'])

models_list = comparison.index.tolist()
colors = ['#EF4444', '#F59E0B', '#10B981']

for i, (metric, vals) in enumerate([
    ('MAPE (%)', comparison['MAPE (%)']),
    ('RMSE', comparison['RMSE']),
    ('R²', comparison['R²']),
]):
    fig.add_trace(go.Bar(x=models_list, y=vals, marker_color=colors,
                         text=[f'{v:.2f}' for v in vals], textposition='outside'),
                  row=1, col=i+1)

fig.update_layout(template='plotly_dark', height=400, showlegend=False,
                  title='Model Performance Comparison')
fig.show()

## 5. Δ-MAPE Analysis

In [ ]:
baseline_mape = comparison.loc['ARIMA (Baseline)', 'MAPE (%)']
print(f'📊 Δ-MAPE Analysis (vs ARIMA Baseline: {baseline_mape}%)')
print('='*50)

for model in models_list[1:]:
    model_mape = comparison.loc[model, 'MAPE (%)']
    delta = baseline_mape - model_mape
    pct_improvement = (delta / baseline_mape) * 100
    print(f'  {model}: {model_mape}% MAPE → Δ = {delta:+.2f}% ({pct_improvement:+.1f}% improvement)')

## 6. Prediction vs Actual

In [ ]:
# XGBoost predictions vs actual
fig = go.Figure()
x_range = list(range(len(y_test)))

fig.add_trace(go.Scatter(x=x_range, y=y_test.values, mode='lines',
                         name='Actual', line=dict(color='#3B82F6', width=2.5)))
fig.add_trace(go.Scatter(x=x_range, y=xgb_result['predictions'], mode='lines',
                         name='XGBoost + TVI', line=dict(color='#EF4444', width=2.5, dash='dash')))

fig.update_layout(template='plotly_dark', height=400,
                  title='XGBoost + TVI: Predicted vs Actual (Test Set)',
                  xaxis_title='Week', yaxis_title='Weekly Sales ($)')
fig.show()

## 7. Feature Importance

In [ ]:
# XGBoost feature importance
fi = xgb_result['feature_importance'].head(20)

fig = go.Figure(go.Bar(
    y=fi['feature'][::-1], x=fi['importance'][::-1],
    orientation='h',
    marker=dict(color=fi['importance'][::-1], colorscale='Viridis')
))

fig.update_layout(template='plotly_dark', height=600,
                  title='XGBoost + TVI — Top 20 Feature Importances',
                  xaxis_title='Importance')
fig.show()

# Highlight TVI features
tvi_features = fi[fi['feature'].str.contains('tvi|spike|trend', case=False, na=False)]
print(f'\n⭐ TVI-related features in top 20: {len(tvi_features)}')
if len(tvi_features) > 0:
    print(tvi_features[['feature', 'importance']].to_string())

## 8. Time-Series Cross-Validation

In [ ]:
# 5-fold time-series CV for XGBoost
X_all = featured_df[feature_cols].fillna(0)
y_all = featured_df['Weekly_Sales']

cv_results = time_series_cv(X_all, y_all, model_type='xgboost', n_splits=5)
print(f'\n📊 CV Results:')
print(cv_results['fold_metrics'])

## 9. Save Models

In [ ]:
# Save trained models
save_model(xgb_result['model'], 'xgboost_tvi', xgb_result['metrics'])
save_model(rf_result['model'], 'random_forest', rf_result['metrics'])
print('\n✅ Models saved to:', config.MODELS_DIR)

## Key Findings

| Model | MAPE | R² | Δ-MAPE |
|-------|------|----|--------|
| ARIMA (Baseline) | See above | — | — |
| Random Forest | See above | — | — |
| **XGBoost + TVI** | **Best** | **Best** | **Best improvement** |

### Next: Results Visualization → **Notebook 04**